# SESSION B — the MoRA fork
### MoRA · the LoRA control · the |E| sweep · DPO

**Settings:** GPU **T4 ×2** · Internet **ON** · Persistence **Variables and Files**

This notebook is pinned to the **`kongds/MoRA` fork**, which is peft 0.9.0 and has **no BOFT**. Its twin, `SESSION A`, is pinned to official peft.

**Never change `ENV` in this notebook.** Installing official peft here would silently overwrite the fork and MoRA would vanish mid-grid.

---

### Why this session exists

MoRA's premise is Chapter 2's hypothesis, in its own words:

> *"the low-rank updating mechanism in LoRA may limit the ability of LLMs to effectively learn and memorize new knowledge"*

KGC with a large entity vocabulary is the natural stress test. The question is not *"is MoRA better?"* but **does the MoRA − LoRA margin GROW with |E|?** MoRA's own paper reports it is *"comparable on other tasks"*, so a flat margin is a **predicted** outcome, not a failed experiment — and Chapter 1 predicts it too, if adaptation installs format rather than knowledge.

★ **`--peft lora` must run here as well**, at exactly the same `--entities` and `--seed` as in Session A. Two peft versions produced your three arms; that LoRA pair is the only thing that shows the version is not the cause.

---

### Environment facts (measured)

* **fp16 + `sdpa`** — fp16 + `eager` returns **NaN** on Qwen2.5.
* **One GPU per job**, or Trainer wraps the model in `DataParallel` and autocast breaks.
* **`torchao` removed** — Kaggle's 0.10.0 blocks the transformers import.
* **`transformers==4.57.6`** — must match Session A exactly, or the LoRA control compares two whole stacks instead of isolating peft.

## 0 · Clone + install

In [ ]:
import socket, urllib.request
for host in ("github.com", "huggingface.co"):
    try:
        print(f"DNS   ok   {host} -> {socket.gethostbyname(host)}")
    except Exception as e:
        print(f"DNS   FAIL {host}: {e}    <-- Settings > Internet > ON")
try:
    print("HTTPS ok   status", urllib.request.urlopen("https://github.com", timeout=10).status)
except Exception as e:
    print("HTTPS FAIL:", e)

In [ ]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
DEST     = "/kaggle/working/repo"

import os, subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated existing clone")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, DEST])
    print("cloned")

os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("\n\u2605 Does that hash match your latest push?")

In [ ]:
# ============ SESSION B — DO NOT CHANGE ============
ENV = "mora"
PIN_TRANSFORMERS = "4.57.6"     # MUST be identical in Session A
# ===================================================

import json, subprocess, sys

def stack():
    code = ("import inspect, json, peft, transformers; print(json.dumps({"
            "'peft': peft.__version__, 'tf': transformers.__version__,"
            "'mora': 'use_mora' in inspect.signature(peft.LoraConfig.__init__).parameters,"
            "'boft': hasattr(peft, 'BOFTConfig')}))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if s and s["mora"] and s["tf"] == PIN_TRANSFORMERS:
    print(f"stack correct: peft {s['peft']} / transformers {s['tf']} — skipping install")
else:
    print(f"installing… (have: {s})")
    !pip install -q -r requirements.txt
    # the fork goes LAST -- it overwrites official peft
    !pip install -q git+https://github.com/kongds/MoRA.git#subdirectory=peft-mora
    !pip install -q "transformers=={PIN_TRANSFORMERS}"
    s = stack()

# torchao 0.10.0 blocks the transformers import; nothing here uses it
!pip uninstall -y -q torchao 2>/dev/null

import peft, transformers
from src.utils.config import peft_env, usable_peft_methods
print(f"\nSESSION B | peft {peft.__version__} | transformers {transformers.__version__}")
print("detected:", peft_env()["peft_env"], "| runnable:", ", ".join(usable_peft_methods()))
assert peft_env()["has_mora"], "MoRA missing — the fork did not install"
assert transformers.__version__ == PIN_TRANSFORMERS, (
    f"transformers {transformers.__version__} != {PIN_TRANSFORMERS}. "
    "Session A and B MUST match or the LoRA control is meaningless.")

## 1 · Data

Chapter 2 needs the big graphs. YAGO3-10 is ~1M triples, so this takes a couple of minutes.

In [ ]:
!python -m scripts.fetch_data --datasets WN18RR YAGO3-10

## 2 · Smoke test — ~5 min

Expect **BOFT to fail** here; it lives in Session A. Everything else must pass, and the script exits 0 when BOFT is the only failure.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m scripts.smoke_test

## 3 · The |E| sweep — Chapter 2's decisive experiment

$|\mathcal{E}|$ is swept **within YAGO3-10** by degree-biased induced subgraph, with the triple budget held fixed. Sweeping across datasets instead would confound $|\mathcal{E}|$ with relation count (237 / 11 / 37), density and label quality.

The quantity of interest is the **slope of the MoRA − LoRA margin against $\log|\mathcal{E}|$**, not either number alone.

★ Eight runs, paired one per T4.

In [ ]:
import subprocess

def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

BASE = "python -m chapters.ch2_adaptation.run --dataset YAGO3-10 --triples 10000 --seed 42"

for E in (10000, 25000, 50000, 123182):
    print(f"\n===== |E| = {E:,} =====")
    pair(f"{BASE} --peft lora --entities {E}",
         f"{BASE} --peft mora --entities {E}")

> The `--peft lora --entities 123182 --seed 42` run above **is** the cross-environment control. Session A runs the identical command under official peft.

### 3b · DPO — Phase 2, on the Phase-1 winner only

KG-LLM trains on `random.choice(all_entities)` — uniformly random, therefore usually type-violating and trivially separable. We replace those with type-consistent near-misses and ask whether preference optimisation reduces structural hallucination.

Verified **0 of 188** papers apply preference optimisation to KGC.

In [ ]:
SFT = "checkpoints/ch2-mora-E123182-T10000-s42"     # <-- edit to the actual winner
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.run --peft dpo --sft-adapter {SFT} \
        --negatives type_consistent --entities 123182 --triples 10000

## 4 · Analysis

Run `verify_env_control` **after** unzipping Session A's `results/` into this session, so both environments' LoRA runs are visible to it.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.analyse --forgetting

In [ ]:
# Upload results_sessionA.zip as a Kaggle Dataset, then:
# !unzip -o /kaggle/input/<your-dataset>/results_sessionA.zip -d /kaggle/working/repo/

# \u2605 Set --tolerance from YOUR seed variance (printed by the cell above),
#   not from the default. A tolerance chosen because it happens to pass is the
#   first thing an examiner will probe.
!python -m scripts.verify_env_control --tolerance 0.01

## 5 · Package results

In [ ]:
!zip -qr /kaggle/working/results_sessionB.zip results/
!du -sh /kaggle/working/results_sessionB.zip
!ls results/ | head -20

---
### Session B checklist

* [ ] commit hash matches your latest push
* [ ] `transformers 4.57.6` — **identical to Session A**
* [ ] `torchao` absent
* [ ] smoke test: only BOFT failed
* [ ] all 4 `|E|` points ran for both LoRA and MoRA
* [ ] `--peft lora --entities 123182 --seed 42` matches Session A's command exactly
* [ ] `verify_env_control` says **pass**
* [ ] `results_sessionB.zip` downloaded

> **A flat MoRA margin is a result, not a failure.** It is what Chapter 1 predicts if adaptation installs format, and MoRA's own paper reports comparable performance on other tasks. The frozen probe from Session A is what makes that reading defensible rather than ambiguous.